In [ ]:
from datetime import datetime, timedelta, UTC

import pandas as pd
import plotly.express as px

from aare.constants import TIME
from aare_influx.field_request import FieldRequest
from aare_influx.remote_existenz_store import RemoteExistenzStore

# Forecasting hourly mean vs end of hour

I'm a big stupid dumdum, apparently. I thought I'd forecast the mean of every hour to get a more stable forecast,
but that doesn't align with the use case the model is intended for. This also applies to air temp, humidity, flow, etc.
For features with a sum, e.g. rainfall and sunshine duration, the sum per aggregate makes sense and should be kept.
For things like wind, it might be smart to have mean (for sustained) and max for peak.

How much different would it be to forecast to the end of the hour? Anything unexpected?

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
store = RemoteExistenzStore()

In [ ]:
tz = "Europe/Zurich"
period = datetime.now(UTC) - timedelta(days=60)
loc = "bern"
target_mean = FieldRequest("hydro", "temperature", "1h", "mean", loc)
target_last = FieldRequest("hydro", "temperature", "1h", "last", loc)
target_first = FieldRequest("hydro", "temperature", "1h", "first", loc)

In [ ]:
df_mean = store.query(period, target_mean)
df_last = store.query(period, target_last)
df_first = store.query(period, target_first)
df = pd.merge(df_mean, df_last, on=TIME, suffixes=("_mean", "_last"))
df = pd.merge(df, df_first.rename(columns={"temperature_bern": "temperature_bern_first"}), on=TIME)
# df[TIME] = df[TIME].dt.tz_convert(tz)
df

In [ ]:
px.line(df, x=TIME, y=["temperature_bern_mean", "temperature_bern_first", "temperature_bern_last"])

In [ ]:
store.client.query_api().query_data_frame("""
import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])
""")

### todo

- analyze these different queries

```
import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> aggregateWindow(every: 1h, fn: first, timeSrc: "_start")
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])


import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])


from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])
```


- maybe do some tests with interpolate.linear, but the problem is, we only want to interpolate in max 1h gaps.
  Could in theory do 2 queries, one with interpolate and then minute == 0 and another without interpolate but agg(first).
  Then use interpolated ones except for places where agg(first) is none (no value in the entire hour).
  Slightly more expensive but it's kinda fancy and I think it might make it more robust (but more magic-y).
  In a first fix, just using agg(first) would probably suffice and is much simpler.


```
import "interpolate"
import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> interpolate.linear(every: 10m)
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])
```


- implement "first" as special case with timeSrc
- implement last, mean, etc. as catchall agg without timeSrc
- implement "exact" with minute == 0
- make script to do historical forecasting on data from postgres db. doesn't have to be pretty yet, just for comparison.
- make tool to show multiple different production forecast alongside 10min measurements for comparison.
- implement bias correction based on weighted linear extrapolation of the last 3-4 10min datapoints up to the full hour we predicted.
  maybe give claude a shot at this; it's simple to describe, but maybe not so simple to implement. testing is a challenge.
- (interactive) tuning of the bias correction parameters using the historical forecasts from the prod db
- refresh oraku2db mirrors to represent the target better (same target as model is trained on: first or if there's time fancy interpolate)
